# Protobuf Basics — 05: Schema Evolution

Protobuf schema evolution is governed by strict rules about field numbers.
Understanding these rules prevents breaking existing producers and consumers.

Topics: adding fields, removing fields, reserved numbers, type compatibility,
backward/forward compatibility, evolution vs Avro.


In [1]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob as glib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/protobuf_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('protobuf-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 21:41:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/10 21:41:38 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark 4.0.2


## Step 1 — Adding Fields: Backward Compatible



In [2]:

print("""
Protobuf evolution rule: field NUMBERS never change.

Adding a field (backward compatible):
  v1:                        v2 (added discount + channel):
  message Order {            message Order {
    int64  order_id = 1;       int64  order_id    = 1;
    string product  = 2;       string product     = 2;
    double price    = 3;       double price       = 3;
    string status   = 4;       string status      = 4;
                               double discount    = 5;  // NEW
                               string channel     = 6;  // NEW
  }                          }

Old reader + new data (has field 5,6):
  → Old reader ignores unknown fields (field 5,6 silently skipped)
  → No error, backward compatible ✅

New reader + old data (no field 5,6):
  → New reader sees 0.0 for discount, "" for channel (default values)
  → Forward compatible ✅

Default values in proto3:
  int32/int64:  0
  double/float: 0.0
  string:       ""
  bool:         false
  message:      null (not set)
  repeated:     empty list
""")



Protobuf evolution rule: field NUMBERS never change.

Adding a field (backward compatible):
  v1:                        v2 (added discount + channel):
  message Order {            message Order {
    int64  order_id = 1;       int64  order_id    = 1;
    string product  = 2;       string product     = 2;
    double price    = 3;       double price       = 3;
    string status   = 4;       string status      = 4;
                               double discount    = 5;  // NEW
                               string channel     = 6;  // NEW
  }                          }

Old reader + new data (has field 5,6):
  → Old reader ignores unknown fields (field 5,6 silently skipped)
  → No error, backward compatible ✅

New reader + old data (no field 5,6):
  → New reader sees 0.0 for discount, "" for channel (default values)
  → Forward compatible ✅

Default values in proto3:
  int32/int64:  0
  double/float: 0.0
  string:       ""
  bool:         false
  message:      null (not set)
  repeated:

## Step 2 — Removing Fields: Reserve the Number



In [3]:

print("""
Removing a field: MUST reserve the number to prevent future reuse.

WRONG (dangerous):
  // v3: removed discount (field 5)
  message Order {
    int64  order_id = 1;
    string product  = 2;
    double price    = 3;
    string status   = 4;
    // field 5 removed — DANGER: someone might add "campaign_id = 5" later!
    string channel  = 6;
  }

CORRECT:
  message Order {
    int64  order_id = 1;
    string product  = 2;
    double price    = 3;
    string status   = 4;
    reserved 5;                  // ← reserve the number
    reserved "discount";         // ← reserve the name too
    string channel  = 6;
  }

Why reserved matters:
  If you add "campaign_id = 5" in v4, old data that had discount=99.99
  will be misread as campaign_id=99.99 → silent data corruption!
  
  reserved 5; prevents: "FieldAlreadyExists" error if anyone tries to reuse 5.
""")



Removing a field: MUST reserve the number to prevent future reuse.

WRONG (dangerous):
  // v3: removed discount (field 5)
  message Order {
    int64  order_id = 1;
    string product  = 2;
    double price    = 3;
    string status   = 4;
    // field 5 removed — DANGER: someone might add "campaign_id = 5" later!
    string channel  = 6;
  }

CORRECT:
  message Order {
    int64  order_id = 1;
    string product  = 2;
    double price    = 3;
    string status   = 4;
    reserved 5;                  // ← reserve the number
    reserved "discount";         // ← reserve the name too
    string channel  = 6;
  }

Why reserved matters:
  If you add "campaign_id = 5" in v4, old data that had discount=99.99
  will be misread as campaign_id=99.99 → silent data corruption!
  
  reserved 5; prevents: "FieldAlreadyExists" error if anyone tries to reuse 5.



## Step 3 — Type Compatibility Rules



In [4]:

print("""
Wire-compatible type changes (safe):
  int32 ↔ int64 ↔ uint32 ↔ uint64 ↔ bool  (all varint, wire type 0)
  string ↔ bytes                             (both length-delimited, wire type 2)
  fixed32 ↔ sfixed32 ↔ float                (all 32-bit, wire type 5)
  fixed64 ↔ sfixed64 ↔ double               (all 64-bit, wire type 1)
  
  embedded message ↔ bytes                  (both wire type 2 — message is just bytes)

Wire-INCOMPATIBLE type changes (data corruption):
  int32 → double  (varint→64-bit)
  string → int32  (length-del→varint)
  float → int64   (32-bit→varint)
  
Practical rules:
  ✅ int32 → int64: safe (old values fit in int64)
  ✅ float → double: safe (old values widen)
  ✅ string → bytes: safe (same wire type)
  ❌ int32 → string: different wire type → corruption
  ❌ double → int32: different wire type + truncation
  ❌ optional → repeated: different semantics (but same wire type, reads as list)
""")



Wire-compatible type changes (safe):
  int32 ↔ int64 ↔ uint32 ↔ uint64 ↔ bool  (all varint, wire type 0)
  string ↔ bytes                             (both length-delimited, wire type 2)
  fixed32 ↔ sfixed32 ↔ float                (all 32-bit, wire type 5)
  fixed64 ↔ sfixed64 ↔ double               (all 64-bit, wire type 1)
  
  embedded message ↔ bytes                  (both wire type 2 — message is just bytes)

Wire-INCOMPATIBLE type changes (data corruption):
  int32 → double  (varint→64-bit)
  string → int32  (length-del→varint)
  float → int64   (32-bit→varint)
  
Practical rules:
  ✅ int32 → int64: safe (old values fit in int64)
  ✅ float → double: safe (old values widen)
  ✅ string → bytes: safe (same wire type)
  ❌ int32 → string: different wire type → corruption
  ❌ double → int32: different wire type + truncation
  ❌ optional → repeated: different semantics (but same wire type, reads as list)



## Step 4 — Protobuf vs Avro Evolution



In [5]:

print("""
Evolution comparison:

  Feature                     Protobuf         Avro
  ──────────────────────────────────────────────────────
  Add field                   ✅ (0/empty def) ✅ (with default)
  Remove field                ✅ (reserve #)   ✅ (with default)
  Rename field                ✅ (# unchanged) ⚠️  (use alias)
  Change type (compatible)    ✅ (wire compat) ✅ (schema resolution)
  Change type (incompatible)  ❌ corruption    ❌ rejected
  Schema in data              ❌ No             ✅ Yes (header)
  Compatibility mode          Manual (rules)   Configured (BACKWARD/FORWARD/FULL)
  Schema registry support     ✅ Confluent SR  ✅ Confluent SR
  
Key difference:
  Avro: compatibility is CHECKED by the Schema Registry (rejects bad schemas)
  Protobuf: compatibility is the DEVELOPER's RESPONSIBILITY (no automatic check)

Best practice with Protobuf:
  1. Always review field numbers before publishing new .proto versions
  2. Never remove a field without reserving its number
  3. Never change field numbers
  4. Use Confluent Schema Registry for central governance
""")



Evolution comparison:

  Feature                     Protobuf         Avro
  ──────────────────────────────────────────────────────
  Add field                   ✅ (0/empty def) ✅ (with default)
  Remove field                ✅ (reserve #)   ✅ (with default)
  Rename field                ✅ (# unchanged) ⚠️  (use alias)
  Change type (compatible)    ✅ (wire compat) ✅ (schema resolution)
  Change type (incompatible)  ❌ corruption    ❌ rejected
  Schema in data              ❌ No             ✅ Yes (header)
  Compatibility mode          Manual (rules)   Configured (BACKWARD/FORWARD/FULL)
  Schema registry support     ✅ Confluent SR  ✅ Confluent SR
  
Key difference:
  Avro: compatibility is CHECKED by the Schema Registry (rejects bad schemas)
  Protobuf: compatibility is the DEVELOPER's RESPONSIBILITY (no automatic check)

Best practice with Protobuf:
  1. Always review field numbers before publishing new .proto versions
  2. Never remove a field without reserving its number
  3. Never chan

## Summary



In [6]:

print("""
Protobuf evolution golden rules:
  1. Field numbers are PERMANENT — never change them
  2. Adding fields → always safe (old readers ignore unknown fields)
  3. Removing fields → MUST use: reserved N; reserved "field_name";
  4. Only change types that are wire-compatible
  5. proto3 default values replace missing fields (0, "", false, [])

Safe changes:
  ✅ Add field (new number)
  ✅ Remove field (+ reserve)
  ✅ Rename field (number stays same)
  ✅ int32 → int64 (both varint)
  ✅ float → double (both fixed-width, different sizes)
  ✅ string → bytes (both length-delimited)

Dangerous changes:
  ❌ Change field number
  ❌ Remove without reserving
  ❌ Incompatible type change (int32 → string)
  ❌ Reuse field number for different type
""")



Protobuf evolution golden rules:
  1. Field numbers are PERMANENT — never change them
  2. Adding fields → always safe (old readers ignore unknown fields)
  3. Removing fields → MUST use: reserved N; reserved "field_name";
  4. Only change types that are wire-compatible
  5. proto3 default values replace missing fields (0, "", false, [])

Safe changes:
  ✅ Add field (new number)
  ✅ Remove field (+ reserve)
  ✅ Rename field (number stays same)
  ✅ int32 → int64 (both varint)
  ✅ float → double (both fixed-width, different sizes)
  ✅ string → bytes (both length-delimited)

Dangerous changes:
  ❌ Change field number
  ❌ Remove without reserving
  ❌ Incompatible type change (int32 → string)
  ❌ Reuse field number for different type

